In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from models.unetpp_3d import UNetPP3D
from utils.unets_helper_functions import (
    set_seed,
    train_one_epoch,
    validate_one_epoch,
    save_checkpoint,
    PatchDataset,
    evaluate_full_volume
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


In [5]:
PATCH_SIZE = 80
PATCHES_PER_CASE = 6

train_dataset = PatchDataset(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        augment=False)

val_dataset = PatchDataset(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=2,    
    pin_memory=True
)

In [6]:
model = UNetPP3D(
    in_channels=1,
    out_channels=7,
    base_filters=16
).to(device)

print("UNet++ Initialized")

UNet++ Initialized


In [7]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
weights = torch.tensor([0.1, 1, 1, 1, 1, 1, 1]).to(device)
ce_loss = nn.CrossEntropyLoss(weight=weights)
scaler = torch.cuda.amp.GradScaler()


C:\Users\dhanu\AppData\Local\Temp\ipykernel_17192\2337973256.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [8]:
EPOCHS = 40

PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "unetpp_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        ce_loss,
        device
    )

    val_loss, val_dice = validate_one_epoch(
        model,
        val_loader,
        ce_loss,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch+1}.pth")
        )
        print("Checkpoint Saved")

100%|██████████| 99/99 [06:15<00:00,  3.79s/it]



Epoch 0
Train Loss: 2.5323
Val Loss: 2.3817
Per Class Dice: [0.21326228 0.08110614 0.01755217 0.09770568 0.20773341 0.11496629]
Saved Best Model


100%|██████████| 99/99 [05:24<00:00,  3.28s/it]



Epoch 1
Train Loss: 2.3489
Val Loss: 2.2711
Per Class Dice: [0.28215537 0.06424229 0.02336434 0.05234946 0.12855055 0.10515272]
Saved Best Model


100%|██████████| 99/99 [05:39<00:00,  3.43s/it]


...


Epoch 37
Train Loss: 0.9649
Val Loss: 1.0179
Per Class Dice: [0.7249639  0.93703436 0.90044845 0.79180059 0.82530532 0.92849108]


100%|██████████| 99/99 [04:27<00:00,  2.71s/it]



Epoch 38
Train Loss: 0.9846
Val Loss: 0.9172
Per Class Dice: [0.815478   0.67810958 0.80788818 0.73203245 0.88476692 0.78494775]
Saved Best Model


100%|██████████| 99/99 [04:39<00:00,  2.83s/it]



Epoch 39
Train Loss: 0.9655
Val Loss: 1.0260
Per Class Dice: [0.73134724 0.94395071 0.90352526 0.53249118 0.9139178  0.93729089]
Checkpoint Saved


In [9]:
print("Training Complete")

Training Complete


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNetPP3D().to(device)

model_path = "../experiments/unetpp_fold0/best_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


Model loaded for testing.


C:\Users\dhanu\AppData\Local\Temp\ipykernel_18528\2368241481.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


In [6]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.7619944738001352), np.float64(0.7279124817811321), np.float64(0.5383188246919954), np.float64(0.6886213947394796), np.float64(0.7246009629891048), np.float64(0.3332003651443021)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.7224811184953327), np.float64(0.6280422951710929), np.float64(0.6565396502876678), np.float64(0.7778071335388074), np.float64(0.8239129170359666), np.float64(0.37735180965051573)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.6793798811797692), np.float64(0.7291597293100457), np.float64(0.5304404755162813), np.float64(0.718290617127553), np.float64(0.7452416693213799), np.float64(0.3461536124658232)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.7335535302521783), np.float64(0.7627708545173623), np.float64(0.5608818

## UNet++ 3D Patch-Based Validation Performance

The model was trained for 40 epochs using patch-based validation.  
Below are the final patch validation metrics obtained at the best validation loss epoch.

### Final Training Statistics (Epoch 38 - Best Model)

- Train Loss: 0.9846  
- Validation Loss: 0.9172  

### Patch-Based Per-Class Dice (Validation)

- Mandible: 0.8155  
- Brainstem: 0.6781  
- Spinal Cord: 0.8079  
- Parotid Left: 0.7320  
- Parotid Right: 0.8848  
- Oral Cavity: 0.7849  


## Observations

- Parotid glands and Mandible achieve high Dice scores (>0.80).
- Spinal Cord performance is strong under patch-based validation.

Overall, the UNet++ model demonstrates effective multi-organ segmentation performance under patch-based validation.


---
# Evaluating on Full Volume 
~ (patch = 80 , kernel = 60)

| Organ ID | Organ (Based on Your Map) | Mean Dice  | Std Dev |
| -------- | ------------------------- | ---------- | ------- |
| 1        | Bone Mandible             | **0.7615** | 0.0455  |
| 2        | Brainstem                 | **0.7197** | 0.0520  |
| 3        | Spinal Cord               | **0.5555** | 0.0460  |
| 4        | Parotid Left              | **0.7037** | 0.0480  |
| 5        | Parotid Right             | **0.7239** | 0.0825  |
| 6        | Oral Cavity               | **0.3755** | 0.0438  |


Full Volume eval Summary

**Strong Organs**
Mandible (0.76) → Very stable and strong
Brainstem (0.72) → Consistent
Parotids (~0.70–0.72) → Good bilateral symmetry

**Moderate**
Spinal Cord (0.55)
This is actually reasonable for 3D patch training
Thin elongated structures are hardest in sliding-window inference

**Weakest**
Oral Cavity (0.37)
Likely due to:
Irregular shape, High anatomical variation ,Lower contrast boundary